# National model comparison - paired bootstrap confidence intervals

The headline RSF / Cox / Weibull AFT comparison rests on a single seed-42 80/20
split. This notebook puts uncertainty bounds on it with a **paired bootstrap over
the shared held-out test set** (the same procedure the PA case study used, ported
national): resample the test rows with replacement, rescore all three models'
*already-computed* risk scores on the identical resample, and read percentile CIs
for each model's C-index and for the pairwise deltas. Models are never refit.

**Inputs** — the risk handoff files written by Notebooks 5 and 6
(`us_rsf_test_risk.csv`, `us_parametric_test_risk.csv`), reconciled 1:1 on bridge
keys, with the recomputed point C-indices asserted against the shipped metrics
JSONs (split-parity proof without reloading any matrix).

**Method notes** (all ported from the PA study's bootstrap cell):
- B = 1000 resamples, `numpy.default_rng(2026)`, one shared index draw per
  iteration -> all three models and all deltas are scored on identical resamples
  (paired).
- Resamples are scored with lifelines' `concordance_index` (sksurv's
  exact scorer is too slow for 3,000 calls). Risk scores are negated for lifelines'
  survival-time orientation, and the two implementations are asserted to agree
  within 2e-3 on the full test set first.
- `p < 0.002` means the delta's sign never flipped in 1000 resamples (2/B).

**Output:** `us_model_bootstrap.json` — per-model 95% CIs + pairwise deltas
(AFT−Cox, AFT−RSF, Cox−RSF) with CIs and two-sided p-values; cited by the README's
model-comparison table.


In [ ]:
# ── Load + reconcile the two risk files, verify against shipped metrics ───────
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sksurv.metrics import concordance_index_censored
from lifelines.utils import concordance_index

RSF_RISK     = Path("us_rsf_test_risk.csv")
PARA_RISK    = Path("us_parametric_test_risk.csv")
RSF_METRICS  = Path("us_rsf_metrics.json")
PARA_METRICS = Path("us_parametric_metrics.json")
OUT_JSON     = Path("us_model_bootstrap.json")

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]
B = 1000
SMOKE_TEST = False   # True -> _smoke inputs/outputs, B = 200

if SMOKE_TEST:
    if Path("us_rsf_test_risk_smoke.csv").exists():
        RSF_RISK     = Path("us_rsf_test_risk_smoke.csv")
        PARA_RISK    = Path("us_parametric_test_risk_smoke.csv")
        RSF_METRICS  = Path("us_rsf_metrics_smoke.json")
        PARA_METRICS = Path("us_parametric_metrics_smoke.json")
    OUT_JSON = OUT_JSON.with_stem(OUT_JSON.stem + "_smoke")
    B = 200
    print(f"SMOKE_TEST: {RSF_RISK} -> {OUT_JSON}, B={B}")

r = pd.read_csv(RSF_RISK,  dtype={k: str for k in KEYS})
p = pd.read_csv(PARA_RISK, dtype={k: str for k in KEYS})
for d in (r, p):
    for k in KEYS:
        d[k] = d[k].str.strip()
assert len(r) == len(p), f"row counts differ: {len(r)} vs {len(p)}"
m = r.merge(p, on=KEYS, how="inner", validate="1:1", suffixes=("", "_p"))
assert len(m) == len(r), "key sets differ between the two risk files"
assert (m["event"] == m["event_p"]).all() and np.allclose(m["time"], m["time_p"]), \
    "event/time disagree between the two files — different splits?"

ev = m["event"].to_numpy(bool)
t  = m["time"].to_numpy(float)
risks = {"rsf": m["rsf_risk"].to_numpy(float),
         "cox": m["cox_risk"].to_numpy(float),
         "aft": m["aft_risk"].to_numpy(float)}

rsf_ref, para_ref = json.loads(RSF_METRICS.read_text()), json.loads(PARA_METRICS.read_text())
expected = {"rsf": rsf_ref["c_index_test"],
            "cox": para_ref["cox_ph"]["c_index_test"],
            "aft": para_ref["weibull_aft"]["c_index_test"]}

# Point estimates must reproduce the shipped numbers, and lifelines' scorer must agree
# with sksurv's before it's trusted for resamples.
point = {}
for name, risk in risks.items():
    ci_exact = float(concordance_index_censored(ev, t, risk)[0])
    assert abs(ci_exact - expected[name]) < 1e-4, \
        f"{name}: recomputed C {ci_exact:.5f} != shipped {expected[name]} — stale handoff files?"
    ci_ll = float(concordance_index(t, -risk, ev))
    assert abs(ci_ll - ci_exact) < 2e-3, f"{name}: concordance implementations disagree"
    point[name] = ci_exact
    print(f"{name}: point C-index {ci_exact:.4f} (matches shipped {expected[name]})")

print(f"\nn_test {len(t):,}   events {int(ev.sum()):,}")


rsf: point C-index 0.7547 (matches shipped 0.7547)


cox: point C-index 0.8835 (matches shipped 0.8835)


aft: point C-index 0.8839 (matches shipped 0.8839)

n_test 194,781   events 57,065


In [2]:
# ── Paired bootstrap: per-model CIs + pairwise deltas from shared resamples ───
rng = np.random.default_rng(2026)
names = ["rsf", "cox", "aft"]
cis = {n: np.empty(B) for n in names}

t0 = time.time()
for b in range(B):
    ii = rng.integers(0, len(t), len(t))
    tb, eb = t[ii], ev[ii]
    for n in names:
        cis[n][b] = concordance_index(tb, -risks[n][ii], eb)
    if (b + 1) % 100 == 0:
        print(f"  {b + 1}/{B} resamples  ({time.time()-t0:.0f}s)", flush=True)

def ci95(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return [round(float(lo), 4), round(float(hi), 4)]

results = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "method": "paired bootstrap over the shared held-out test set; models not refit",
    "B": B, "seed": 2026,
    "n_test": int(len(t)), "n_test_events": int(ev.sum()),
    "models": {n: {"c_index": round(point[n], 4), "ci95": ci95(cis[n])} for n in names},
    "pairwise_deltas": {},
}
for hi_m, lo_m in [("aft", "cox"), ("aft", "rsf"), ("cox", "rsf")]:
    d = cis[hi_m] - cis[lo_m]                        # paired: same resample indices
    p_two = 2 * min(float((d <= 0).mean()), float((d >= 0).mean()))
    results["pairwise_deltas"][f"{hi_m}_minus_{lo_m}"] = {
        "delta_point": round(point[hi_m] - point[lo_m], 4),
        "ci95": ci95(d),
        "p_two_sided": f"< {2 / B:g}" if p_two == 0 else round(p_two, 4),
    }

OUT_JSON.write_text(json.dumps(results, indent=2))
print(f"\nsaved {OUT_JSON}")


  100/1000 resamples  (193s)


  200/1000 resamples  (386s)


  300/1000 resamples  (580s)


  400/1000 resamples  (773s)


  500/1000 resamples  (966s)


...

  700/1000 resamples  (1352s)


  800/1000 resamples  (1546s)


  900/1000 resamples  (1739s)


  1000/1000 resamples  (1933s)



saved us_model_bootstrap.json


In [3]:
# ── Verification + readable summary ───────────────────────────────────────────
res = json.loads(OUT_JSON.read_text())

print(f"{'model':<6} {'C-index':>8}   95% CI")
for n in ("aft", "cox", "rsf"):
    mdl = res["models"][n]
    lo, hi = mdl["ci95"]
    assert lo <= mdl["c_index"] <= hi, f"{n}: CI does not bracket the point estimate"
    print(f"{n:<6} {mdl['c_index']:>8.4f}   [{lo:.4f}, {hi:.4f}]")

print(f"\n{'delta':<16} {'point':>8}   95% CI              p (two-sided)")
for k, d in res["pairwise_deltas"].items():
    hi_m, lo_m = k.split("_minus_")
    expect = round(res["models"][hi_m]["c_index"] - res["models"][lo_m]["c_index"], 4)
    assert abs(d["delta_point"] - expect) <= 1e-4, f"{k}: delta != C-index difference"
    lo, hi = d["ci95"]
    assert lo <= d["delta_point"] <= hi, f"{k}: CI does not bracket the point delta"
    sig = "" if lo <= 0 <= hi else "  (excludes 0)"
    print(f"{k:<16} {d['delta_point']:>+8.4f}   [{lo:+.4f}, {hi:+.4f}]  {d['p_two_sided']}{sig}")

print("\nAll bootstrap verification checks passed.")


model   C-index   95% CI
aft      0.8839   [0.8825, 0.8851]
cox      0.8835   [0.8822, 0.8847]
rsf      0.7547   [0.7525, 0.7570]

delta               point   95% CI              p (two-sided)
aft_minus_cox     +0.0004   [+0.0001, +0.0006]  < 0.002  (excludes 0)
aft_minus_rsf     +0.1292   [+0.1275, +0.1309]  < 0.002  (excludes 0)
cox_minus_rsf     +0.1288   [+0.1271, +0.1305]  < 0.002  (excludes 0)

All bootstrap verification checks passed.
